In [1]:
# This test program aims to calculate the attenuation factors using 1000 MC sample points and for one .nxspe file.
# This program works for single (convex) crystals.

using FileIO
using MeshIO
using GeometryBasics
using BenchmarkTools
using SparseArrays
using StaticArrays
using LinearAlgebra
include("Modules/sampling.jl")
using .sampling
include("Modules/nxspe.jl")
using .nxspe
include("Modules/sin_crystal.jl")
using .sin_crystal

In [2]:
# Extracting the contents of the .nxspe file.

en_i, azi, pol, data, Δen = nxspe.extract("test_nxspe_data/LET104215_3.7meV_1to1.nxspe")

(3.7f0, Float32[-137.16072, -137.39026, -137.62152, -137.85446, -138.08911, -138.32544, -138.5635, -138.80327, -139.04478, -139.28812  …  41.245438, 41.48468, 41.72211, 41.957756, 42.191696, 42.42389, 42.654366, 42.883137, 43.110214, 43.335594], Float32[48.282505, 48.177177, 48.07187, 47.966606, 47.861397, 47.756275, 47.65122, 47.54627, 47.441406, 47.336624  …  131.69145, 131.58684, 131.4822, 131.3775, 131.27277, 131.16801, 131.06323, 130.95845, 130.85367, 130.74889], Float32[NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN; NaN NaN … NaN NaN], Float32[-2.96, -2.9415, -2.923, -2.9045, -2.886, -2.8675, -2.849, -2.8305, -2.812, -2.7935  …  2.7935, 2.812, 2.8305, 2.849, 2.8675, 2.886, 2.9045, 2.923, 2.9415, 2.96])

In [3]:
# Calculating ki, n_bins, n_detectors from the extracted content of the .nxspe file.


# Finding the initial wavevector, in Angstrom^-1, from the initial energy.
ki = zeros(3)
ki[1] = nxspe.magk_calc(en_i)
# Converting ki to a static array.
ki = SVector{3, Float32}(ki)

# Extracting the number of energy bins and number of detectors.
const n_bins = length(Δen) - 1
const n_detectors = length(azi)

98304

In [4]:
# Calculating the final neutron energy, in meV, for each energy bin.

ef_bins = nxspe.ef_calc(en_i, Δen, n_bins)

320-element Vector{Float32}:
 6.65075
 6.63225
 6.6137505
 6.59525
 6.57675
 6.5582504
 6.53975
 6.52125
 6.5027504
 6.48425
 ⋮
 0.89724994
 0.8787501
 0.86025023
 0.8417499
 0.82325006
 0.8047502
 0.7862499
 0.76775
 0.7492502

In [5]:
# Calculating the final wavevector, in Angstrom^-1, for each energy bin and each detector.

kx, ky, kz = nxspe.kf_calc(ef_bins, pol, azi)

(Float32[-0.98057234 -0.9825924 … 0.9892723 0.987179; -0.9792076 -0.98122483 … 0.9878954 0.98580503; … ; -0.3331611 -0.3338474 … 0.336117 0.33540577; -0.32912263 -0.32980064 … 0.33204272 0.3313401], Float32[-0.9092696 -0.9038486 … 0.9260755 0.93142897; -0.90800405 -0.9025906 … 0.9247866 0.93013257; … ; -0.30893514 -0.30709326 … 0.31464514 0.31646404; -0.30519032 -0.3033708 … 0.31083113 0.31262797], Float32[1.1921976 1.1946537 … -1.1719011 -1.1694212; 1.1905382 1.192991 … -1.1702701 -1.1677935; … ; 0.40506324 0.40589777 … -0.3981673 -0.3973247; 0.40015322 0.40097764 … -0.39334086 -0.39250848])

In [6]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


stl = load("crystal.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

2142

In [7]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, and V1s specifically in preparation for the Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

v1s, e2s, e3s = sampling.ve_calc(vertices, indices)

(SVector{3, Float32}[[13.304199, 20.258078, 46.79252], [13.590828, 20.380196, 46.776127], [13.963831, 20.172203, 46.80407], [13.192872, 20.479755, 46.77794], [13.428433, 20.525827, 46.78789], [13.832438, 20.393225, 46.786858], [12.922315, 20.37644, 46.789036], [13.590828, 20.380196, 46.776127], [13.192872, 20.479755, 46.77794], [13.025568, 20.651573, 46.79539]  …  [14.65111, 20.228249, 49.119], [14.667707, 20.486109, 49.11232], [15.205641, 20.177717, 49.115513], [15.644693, 20.509438, 49.081673], [14.14268, 20.198109, 49.116863], [14.123109, 20.603514, 49.111305], [14.391234, 20.406094, 49.118073], [14.956343, 20.40861, 49.111523], [14.391234, 20.406094, 49.118073], [13.824862, 20.41183, 49.098145]], SVector{3, Float32}[[0.28662872, 0.122117996, -0.016391754], [0.24161053, 0.0130290985, 0.010730743], [-0.45956707, -0.039129257, 0.010292053], [0.23556137, 0.046072006, 0.009952545], [0.16239452, -0.14563179, -0.011764526], [0.25931644, 0.02025795, 0.012813568], [-0.32102966, -0.09597397,

In [8]:
# Setting the desired number of MC sample points and creating vector for coordinates.

const n_mc = 10
mc_coords = Vector{SVector{3, Float32}}(undef, n_mc)

10-element Vector{SVector{3, Float32}}:
 [-1.581024f12, 6.87f-43, -1.5810282f12]
 [6.87f-43, -8.718789f11, 6.87f-43]
 [-1.5810324f12, 6.87f-43, -1.5810366f12]
 [6.87f-43, -1.5808059f12, 6.87f-43]
 [-8.718789f11, 6.87f-43, 2.3181717f35]
 [4.5914f-41, -8.718789f11, 6.87f-43]
 [-8.718789f11, 6.87f-43, 1.0f-45]
 [0.0, NaN, NaN]
 [1.0f-45, 1.0f-45, -3.7968696f34]
 [6.87f-43, -3.8423242f34, 6.87f-43]

In [9]:
# Setting the (estimated) parameters of the sample.

# The reference attenuation coefficent at 25.3 meV in cm^-1.
const μ_ref = Float32(1)
const en_ref = Float32(25.3)

25.3f0

In [10]:
# Calculating the pre-scattering attenuation coefficent.

const μi = sin_crystal.μ_calc(μ_ref, en_ref, en_i)

2.6149259f0

In [11]:
# Generating the sample points.

# Calculating the extrema of the axis-aligned bounding box around the sample.
ranges = sampling.aabb_3d(vertices)
sampling.sample!(ranges, e2s, e3s, v1s, mc_coords, n_faces, n_mc)

10-element Vector{SVector{3, Float32}}:
 [14.510693, 19.579115, 47.998035]
 [16.883198, 21.444386, 47.721344]
 [15.89241, 20.876135, 48.908768]
 [19.06539, 18.25391, 48.434246]
 [18.5391, 18.250303, 47.711967]
 [17.552004, 19.971344, 48.425816]
 [18.98812, 20.220203, 47.277096]
 [16.136698, 21.636196, 48.079525]
 [16.844114, 21.511917, 48.33164]
 [18.913107, 20.717861, 48.588814]

In [12]:
# The pre-scattering neutron path lengths are dependent only on the MC coordinates.
# They can, therefore, be calculated and stored.

len_i = sin_crystal.len_i_calc(e2s, e3s, v1s, mc_coords, n_faces, n_mc)

10-element Vector{Float32}:
 1.7793335
 4.3607726
 3.274442
 1.9037995
 1.9980702
 4.8600583
 6.328455
 3.6477268
 4.4225483
 6.3594036

In [13]:
# Converting the grid of data and attenuation factors to sparse arrays.

# Replacing every NaN value with zero to allow conversion to sparse matrix.
data_copy = copy(data)
data_copy .= ifelse.(isnan.(data_copy), 0, data)
s_data = sparse(data_copy)

320×98304 SparseMatrixCSC{Float32, Int64} with 317849 stored entries:
⎡⣷⣤⣇⣾⣮⣴⣿⣿⣼⣦⣗⣷⣷⣷⣴⣧⣷⣴⣦⣼⣆⣛⣽⣖⣺⣶⣀⣔⣰⣂⣠⣶⣕⣶⣾⣧⣱⣔⣷⣖⎤
⎣⣿⢉⣛⢿⣿⣿⣿⣿⣿⣻⣿⣿⣿⣿⡿⣿⡻⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⣿⣿⢿⡿⣿⢿⣿⣿⣿⎦

In [14]:
# Testing the time taken to output this grid of attenuation factors.

atten_grid = sin_crystal.a_grid_calc(s_data, kx, ky, kz, ef_bins, v1s, e2s, e3s, mc_coords, len_i, n_bins, n_detectors, n_faces, n_mc, μ_ref, en_ref, μi)
# Testing the same (known to be non-zero) datapoint.
display(atten_grid[160,6])
# Converting the attenuation factors to a sparse matrix.
s_atten = sparse(atten_grid)
display(s_atten)

0.0006824521f0

320×98304 SparseMatrixCSC{Float32, Int64} with 317849 stored entries:
⎡⣷⣤⣇⣾⣮⣴⣿⣿⣼⣦⣗⣷⣷⣷⣴⣧⣷⣴⣦⣼⣆⣛⣽⣖⣺⣶⣀⣔⣰⣂⣠⣶⣕⣶⣾⣧⣱⣔⣷⣖⎤
⎣⣿⢉⣛⢿⣿⣿⣿⣿⣿⣻⣿⣿⣿⣿⡿⣿⡻⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⣿⣿⢿⡿⣿⢿⣿⣿⣿⎦

In [15]:
# Benchmarking len_calc() with the first final wavevector and the first MC sampling point.


d_test = SVector{3, Float32}(kx[1][1], ky[1][1], kz[1][1])
d_test = d_test / norm(d_test)
p_test = Vector{SVector{3, Float32}}(undef, n_faces)
det_test = Vector{Float32}(undef, n_faces)
sin_crystal.pdet_calc!(d_test, e2s, e3s, p_test, det_test, n_faces)
test = mc_coords[1]


@benchmark sin_crystal.len_calc(e2s, e3s, d_test, p_test, det_test, test, v1s, n_faces)

BenchmarkTools.Trial: 10000 samples with 9 evaluations per sample.
 Range (min … max):  2.522 μs …  65.900 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.756 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.904 μs ± 895.526 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅▅▇▇█▇▆▃▂                                       ▂▂          ▂
  ██████████▇▇█▆█▇██▇▆▆▅▇▅▆▅▅▆▆▅▁▅▅▆▅▅▆▅▄▅▄▆▅▅▆▅▆███▇██▇▆▅▆▅▄ █
  2.52 μs      Histogram: log(frequency) by time      5.64 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [16]:
# Benchmarking atten_calc() for the first final wavevector and first final energy.


d_test = SVector{3, Float32}(kx[1][1], ky[1][1], kz[1][1])
d_test = d_test / norm(d_test)
p_test = Vector{SVector{3, Float32}}(undef, n_faces)
det_test = Vector{Float32}(undef, n_faces)
sin_crystal.pdet_calc!(d_test, e2s, e3s, p_test, det_test, n_faces)
en_test = ef_bins[1]


@benchmark sin_crystal.atten_calc(d_test, en_test, v1s, e2s, e3s, mc_coords, len_i, p_test, det_test, n_faces, n_mc, μ_ref, en_ref, μi)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  52.400 μs … 590.700 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     66.200 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   71.533 μs ±  18.139 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▁▅▆▄▂▄▅▆███▇▇▇▇▅▃▂▁▁   ▁▂                 ▁▂▁  ▁▂▂▂▂▂▁▁      ▃
  ██████████████████████████▇▇▇▆▅▇▅█▇▇▇▇▇█▇██████████████▆▇▇██ █
  52.4 μs       Histogram: log(frequency) by time       130 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [17]:
# Benchmarking a_grid_calc().


@benchmark sin_crystal.a_grid_calc(s_data, kx, ky, kz, ef_bins, v1s, e2s, e3s, mc_coords, len_i, n_bins, n_detectors, n_faces, n_mc, μ_ref, en_ref, μi)

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 19.756 s (0.00% GC) to evaluate,
 with a memory estimate of 126.10 MiB, over 18 allocations.